# Zodiac GPU Swarm Solver (v3.2 - Fix & Scale)
**Autonomous Multi-Agent Simulated Annealing**

**Intelligence Layer:**
*   **1 Million+ Agents:** Verified scaling for A100 VRAM.
*   **Basin Escape:** Dynamic temperature and swarm expansion.
*   **Fixed:** Restored `get_accuracy` method.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import time
import json
import os
from pathlib import Path

Z408_CIPHERTEXT = r"HER>pl^VPk|1LTG2dNp+B(#TMOZSyT+@nQFHZOPWNYBkBVcoGDpKI(L#K!qY!emN6SCcM+Ak#@NyDBPORAU+RJ-ocLM\TMDH!P.Tg)EHM!2bjWVT4kFB-cyA64M+R_FlWB|y0E|FBc(WCV2GIH-NTtZBK(LW7V#5T+L1*-dW0VlH.+iAyBX1*?WB|^GyF-5lqG2HCoT4M-LMl++>kRiAT4@dB(ARO+rNcFRNpkA-yBk0R(JEPnp^FlWBy1PpL+\R+qjFiNX1V-FBWA+5BPORAU+R(WCV2GIH-NTtZBK(LWVlF+ML.0Sy+@dXNyMZBcRBdNHU\BGkF=N+|GBcA-J.2t#PWBN^FBV0|cE|F-5LR-ckdW@P+|FlF45_oR+|J?VOKWHpLI(LB\AVlH9pZHB?XWVc"
Z408_KNOWN_PLAINTEXT = "ILIKEKILLINGPEOPLEBECAUSEITISSOMUCHFUNITISMOREFUNTHANKILLINGWILDGAMEINTHEFORRESTBECAUSEMANISTHEMOSTDANGEROUSANIMALOFALLTOKILLSOMETHINGGIVESMETHEMOSTTHRILLINGEXPERENCEITISEVENBETTERTHANGETTINGYOURROCKSOFFWITHAGIRLTHEBESTPARTOFITISTHAEWHENIDIEIWILLBEREBORNINPARADICEANDALLTHEIHAVEKILLEDWILLBECOMEMYSLAVESIWILLNOTGIVEYOUMYNAMEBECAUSEYOUWILLTRYTOSLOIDOWNORATOPMYCOLLECTIOGOFSLAVESFORMYAFTERLIFEEBEORIETEMETHHPITI"

QUADGRAMS = {"THER": -3.20, "TION": -3.30, "ATIO": -3.40, "NTHE": -3.45, "OFTH": -3.50, "INGS": -3.55, "IONS": -3.60, "EVER": -3.65, "ATER": -3.68, "ANDT": -3.70, "NTER": -3.72, "OTHE": -3.74, "HERE": -3.76, "FROM": -3.78, "INTO": -3.80, "ESTH": -3.82, "TICS": -3.84, "ITHE": -3.86, "FORM": -3.88, "WHER": -3.90, "EARS": -3.92, "MENT": -3.94, "AINT": -3.96, "ALLY": -3.98, "COUR": -4.00, "LITT": -4.02, "PEOP": -4.04, "OUGH": -4.06, "ENCE": -4.08, "ESTE": -4.10, "RENT": -4.12, "OUSE": -4.14, "ONLY": -4.16, "ABOU": -4.18, "INTH": -4.20, "ITHA": -4.22, "ATHE": -4.24, "ELIK": -4.28, "EAST": -4.30, "WITH": -4.32, "BEEN": -4.34, "BOUT": -4.36, "TURN": -4.38, "AFTE": -4.40, "OVER": -4.42, "ETHA": -4.44, "NTOF": -4.46, "ENTL": -4.48, "RTHE": -4.50, "WOUL": -4.52}
QUAD_FLOOR, BI_FLOOR = -8.0, -5.0
BIGRAMS = {"TH": -1.40, "HE": -1.46, "IN": -1.62, "ER": -1.66, "AN": -1.68, "RE": -1.74, "ON": -1.79, "AT": -1.82, "EN": -1.87, "ND": -1.89, "TI": -1.91, "ES": -1.93, "OR": -1.94, "TE": -1.97, "OF": -1.98, "ED": -2.00, "IS": -2.01, "IT": -2.04, "AL": -2.06, "AR": -2.07, "ST": -2.10, "TO": -2.10, "NT": -2.12, "NG": -2.14, "SE": -2.16, "HA": -2.17, "AS": -2.20, "OU": -2.22, "IO": -2.23, "LE": -2.25, "VE": -2.27, "CO": -2.30, "ME": -2.32, "DE": -2.33, "HI": -2.34, "RI": -2.36, "RO": -2.38, "IC": -2.40, "NE": -2.42, "EA": -2.44, "RA": -2.46, "CE": -2.48, "LI": -2.50, "CH": -2.52, "LL": -2.54, "BE": -2.56, "MA": -2.58, "SI": -2.60, "OM": -2.62, "UR": -2.64}
KNOWN_P_TENSOR = torch.tensor([ord(c) - ord('A') for c in Z408_KNOWN_PLAINTEXT], dtype=torch.int16)


def resolve_checkpoint_path(filename="zodiac_gpu_best.json"):
    """Use Google Drive in Colab so runtime resets do not wipe progress."""
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
        try:
            from google.colab import drive  # type: ignore
            drive.mount('/content/drive', force_remount=False)
            base_dir = Path('/content/drive/MyDrive/zodiac-attack')
            base_dir.mkdir(parents=True, exist_ok=True)
            return str(base_dir / filename)
        except Exception as exc:
            print(f"[warn] Drive mount failed, falling back to local runtime: {exc}")
    return filename


# Dynamo index dtype rule: index tensors must be long/int32/byte/bool — NOT int16.
# cipher_indices and sym_a/sym_b use int32 (index role).
# mappings and new_letters use int16 (data role, 4× smaller than int64).
def _score_fn(mappings, cipher_indices, quad_table, bi_table, inv_quad_len, inv_bi_len):
    plaintexts = mappings[:, cipher_indices].long()
    i0, i1, i2, i3 = plaintexts[:, :-3], plaintexts[:, 1:-2], plaintexts[:, 2:-1], plaintexts[:, 3:]
    quad_scores = quad_table[i0, i1, i2, i3].sum(dim=1) * inv_quad_len
    j0, j1 = plaintexts[:, :-1], plaintexts[:, 1:]
    bi_scores = bi_table[j0, j1].sum(dim=1) * inv_bi_len
    return 0.75 * quad_scores + 0.25 * bi_scores

try:
    _score_fn = torch.compile(_score_fn)
    print("[*] torch.compile active on score kernel")
except Exception:
    pass

class ZodiacGPUSolver:
    def __init__(self, ciphertext, out_path=None, device='cuda'):
        self.device = device if torch.cuda.is_available() else 'cpu'
        self.out_path = out_path or resolve_checkpoint_path("zodiac_gpu_best.json")
        print(f"[*] Checkpoint path: {self.out_path}")
        self.ciphertext, self.symbols = ciphertext, sorted(list(set(ciphertext)))
        self.num_symbols = len(self.symbols)
        self.sym_to_idx = {s: i for i, s in enumerate(self.symbols)}
        self.idx_to_sym = {i: s for s, i in self.sym_to_idx.items()}
        # int32: Dynamo-safe index dtype, 2× smaller than int64 (values 0–69)
        self.cipher_indices = torch.tensor(
            [self.sym_to_idx[c] for c in ciphertext], dtype=torch.int32, device=self.device)
        self.cipher_len = len(self.cipher_indices)
        self.known_p_gpu = KNOWN_P_TENSOR.to(self.device)

        self.quad_table = torch.full((26, 26, 26, 26), QUAD_FLOOR, device=self.device)
        for q, score in QUADGRAMS.items():
            idx = [ord(c) - ord('A') for c in q]
            self.quad_table[idx[0], idx[1], idx[2], idx[3]] = score
        self.bi_table = torch.full((26, 26), BI_FLOOR, device=self.device)
        for b, score in BIGRAMS.items():
            idx = [ord(c) - ord('A') for c in b]
            self.bi_table[idx[0], idx[1]] = score

        self.inv_quad_len = 1.0 / (self.cipher_len - 3)
        self.inv_bi_len   = 1.0 / (self.cipher_len - 1)

        self.swarm_size = self.auto_scale_swarm()
        print(f"[*] Initializing {self.swarm_size} agents on {self.device}")
        # int16 for mapping values (0–25): 4× smaller than int64, valid as data (never used as index)
        self.mappings = torch.randint(0, 26, (self.swarm_size, self.num_symbols),
                                      dtype=torch.int16, device=self.device)
        self.rows = torch.arange(self.swarm_size, device=self.device)
        self._rebuild_temp_mult()
        self.best_score, self.best_mapping = -float('inf'), None
        self.resumed = False
        self.load_state()

    def _rebuild_temp_mult(self):
        s3 = self.swarm_size // 3
        self.temp_mult = torch.ones(self.swarm_size, device=self.device)
        self.temp_mult[:s3]   = 2.0
        self.temp_mult[2*s3:] = 0.25

    def auto_scale_swarm(self):
        if self.device == 'cpu': return 2048
        torch.cuda.empty_cache()
        total_mem = torch.cuda.get_device_properties(0).total_memory
        # int16 mappings×2 (70×2×2=280) + plaintexts long (408×8=3264) + score floats (405×4=1620) ≈ 5164
        safe_size = int(total_mem * 0.5 / 6144)
        return max(16384, min(safe_size, 2097152))

    def get_accuracy(self, mapping):
        plain = mapping[self.cipher_indices]
        return (plain == self.known_p_gpu).float().mean().item()

    def load_state(self):
        if os.path.exists(self.out_path):
            with open(self.out_path, 'r') as f:
                data = json.load(f)
                self.best_score = data['best_score']
                mapping_dict = data['best_mapping']
                loaded_mapping = torch.zeros(self.num_symbols, dtype=torch.int16, device=self.device)
                for sym, char in mapping_dict.items():
                    if sym in self.sym_to_idx:
                        loaded_mapping[self.sym_to_idx[sym]] = ord(char) - ord('A')
                self.best_mapping = loaded_mapping
                seed_n = (self.swarm_size * 3) // 4
                self.mappings[:seed_n] = loaded_mapping
                noise = torch.randint(0, 26, (seed_n, self.num_symbols),
                                      dtype=torch.int16, device=self.device)
                mask  = torch.rand((seed_n, self.num_symbols), device=self.device) < 0.05
                self.mappings[:seed_n] = torch.where(mask, noise, self.mappings[:seed_n])
                self.resumed = True
                print(f"[resume] Loaded prior best (score: {self.best_score:.4f})")

    def save_state(self):
        mapping_dict = {self.idx_to_sym[i]: chr(ord('A') + self.best_mapping[i].item())
                        for i in range(self.num_symbols)}
        plaintext = "".join([chr(ord('A') + self.best_mapping[idx].item())
                             for idx in self.cipher_indices])
        out_parent = os.path.dirname(self.out_path)
        if out_parent:
            os.makedirs(out_parent, exist_ok=True)
        with open(self.out_path, 'w') as f:
            json.dump({"best_score": self.best_score, "best_mapping": mapping_dict,
                       "best_plaintext": plaintext,
                       "updated_at": time.strftime("%Y-%m-%d %H:%M:%S")}, f, indent=2)

    def calculate_scores(self, mappings):
        return _score_fn(mappings, self.cipher_indices, self.quad_table, self.bi_table,
                         self.inv_quad_len, self.inv_bi_len)

    def solve(self, iterations=15000, temp_start=0.5, temp_end=0.005):
        scores = self.calculate_scores(self.mappings)
        t_factor = (temp_end / temp_start) ** (1.0 / iterations)
        temp_schedule = [temp_start * (t_factor ** i) for i in range(iterations)]
        t0 = time.time()

        for i in range(iterations):
            T = temp_schedule[i]
            rand_val = torch.rand(self.swarm_size, device=self.device)
            # int32: Dynamo-safe for scatter index ops, 2× smaller than int64
            sym_a = torch.randint(0, self.num_symbols, (self.swarm_size,),
                                  dtype=torch.int32, device=self.device)
            sym_b = torch.randint(0, self.num_symbols, (self.swarm_size,),
                                  dtype=torch.int32, device=self.device)

            orig_a = self.mappings[self.rows, sym_a]
            orig_b = self.mappings[self.rows, sym_b]

            new_mappings  = self.mappings.clone()
            mask_reassign = rand_val >= 0.7
            mask_swap     = ~mask_reassign

            new_letters = torch.randint(0, 26, (self.swarm_size,),
                                        dtype=torch.int16, device=self.device)
            new_mappings[self.rows[mask_reassign], sym_a[mask_reassign]] = new_letters[mask_reassign]
            new_mappings[self.rows[mask_swap], sym_a[mask_swap]] = orig_b[mask_swap]
            new_mappings[self.rows[mask_swap], sym_b[mask_swap]] = orig_a[mask_swap]

            new_scores  = self.calculate_scores(new_mappings)
            delta       = new_scores - scores
            accept_rand = torch.rand(self.swarm_size, device=self.device)
            eff_T       = T * self.temp_mult
            accept_mask = (delta > 0) | (accept_rand < torch.exp(delta / eff_T))
            self.mappings[accept_mask] = new_mappings[accept_mask]
            scores[accept_mask]        = new_scores[accept_mask]

            best_idx = torch.argmax(scores)
            if scores[best_idx] > self.best_score:
                self.best_score   = scores[best_idx].item()
                self.best_mapping = self.mappings[best_idx].clone()
                self.save_state()

            if i % 1000 == 0:
                if self.best_mapping is not None:
                    self.save_state()
                acc = self.get_accuracy(self.best_mapping if self.best_mapping is not None else self.mappings[0])
                print(f"[Iter {i:5d}] Best: {self.best_score:.4f} | Acc: {acc*100:4.1f}% | Swarm: {self.swarm_size} | T: {T:.4f} | {time.time()-t0:.1f}s")

        return self.best_score

    def expand_swarm(self):
        new_size = self.swarm_size * 2
        print(f"[!] Expanding Swarm: {self.swarm_size} -> {new_size}")
        new_mappings = torch.randint(0, 26, (self.swarm_size, self.num_symbols),
                                     dtype=torch.int16, device=self.device)
        if self.best_mapping is not None:
            new_mappings[:] = self.best_mapping
            noise = torch.randint(0, 26, (self.swarm_size, self.num_symbols),
                                  dtype=torch.int16, device=self.device)
            mask  = torch.rand(new_mappings.shape, device=self.device) < 0.15
            new_mappings = torch.where(mask, noise, new_mappings)
        self.mappings   = torch.cat([self.mappings, new_mappings], dim=0)
        self.swarm_size = new_size
        self.rows       = torch.arange(self.swarm_size, device=self.device)
        self._rebuild_temp_mult()

if __name__ == "__main__":
    solver = ZodiacGPUSolver(Z408_CIPHERTEXT)
    last_best = solver.best_score if solver.resumed else -float('inf')
    plateau_rounds = 0
    while True:
        heat_f = 2 ** (plateau_rounds // 2)
        if plateau_rounds > 6:
            solver.expand_swarm()
            plateau_rounds = 0
        t_start = 0.5 * heat_f
        if solver.resumed and plateau_rounds == 0:
            t_start = min(t_start, 0.05)
            solver.resumed = False
        if heat_f > 1: print(f"[!] Basin Escape Mode: Temp {t_start:.2f}")
        res = solver.solve(iterations=15000, temp_start=t_start)
        if res > last_best + 0.0001: last_best, plateau_rounds = res, 0
        else: plateau_rounds += 1
        print(f"[*] Round Done. Best: {last_best:.4f}. Plateau: {plateau_rounds}")


In [ ]:
# Checkpoint inspector (run before training)
ckpt_path = resolve_checkpoint_path("zodiac_gpu_best.json")
print(f"[*] Active checkpoint: {ckpt_path}")

if os.path.exists(ckpt_path):
    with open(ckpt_path, "r") as f:
        ckpt = json.load(f)
    best_score = ckpt.get("best_score")
    updated_at = ckpt.get("updated_at", "unknown")
    preview = ckpt.get("best_plaintext", "")[:120]
    print(f"[resume-check] Found checkpoint")
    print(f"  updated_at: {updated_at}")
    print(f"  best_score: {best_score}")
    print(f"  plaintext_preview: {preview}...")
else:
    print("[resume-check] No checkpoint found yet. First run will create one.")